### 📊 Experiments Results

🔹 1. Imports

In [40]:
import json
import pandas as pd
from pathlib import Path

🔹 2. Função de interpretação

In [41]:
def interpret(value, metric):
    if metric == "accuracy":
        if value > 0.8: return "🔥 Muito bom"
        elif value > 0.6: return "👍 Ok"
        else: return "⚠️ Baixo"
    
    if metric == "proxy_score":
        if value > 0.75: return "🔥 Forte"
        elif value > 0.5: return "👍 Médio"
        else: return "⚠️ Fraco"

🔹 3. Load + processamento

In [ ]:
def load_and_process(path):
    with open(path, "r") as f:
        data = json.load(f)

    rows = []

    for item in data:
        pred = item.get("prediction")
        gold = item.get("gold_label")

        steps = item.get("steps", [])
        evidence_counts = [len(s.get("evidence", [])) for s in steps]

        avg_evidence = sum(evidence_counts) / len(evidence_counts) if evidence_counts else 0

        rows.append({
            "claim_id": item.get("id") or item.get("claim"),
            "prediction": pred,
            "label": gold,
            "correct": pred == gold,
            "num_steps": len(steps),
            "avg_evidence_per_step": avg_evidence,
        })

    df = pd.DataFrame(rows)

    # métricas
    accuracy = df["correct"].mean()
    evidence_score = df["avg_evidence_per_step"].mean()
    proxy_score = 0.7 * accuracy + 0.3 * min(evidence_score / 5.0, 1.0)

    metrics = {
        "accuracy": accuracy,
        "proxy_score": proxy_score
    }

    return df, metrics

🔹 4. Paths

In [43]:
paths = {
    "baseline": [
        "../outputs/averitec_original_pipeline/2026-03-24_19-33-17/averitec_dev.json",
    ],
    "iterative": [
        "../outputs/averitec_iterative_pipeline/2026-03-24_19-41-59/averitec_dev.json"
    ]
}

🔹 5. Carregamento geral

In [44]:
dfs = {}
results = {}

for pipeline, file_list in paths.items():
    dfs[pipeline] = []
    results[pipeline] = []

    for path in file_list:
        df, metrics = load_and_process(path)

        run_name = Path(path).parent.name
        df["run"] = run_name

        dfs[pipeline].append(df)
        results[pipeline].append(metrics)

🔹 6. Concatenar tudo

In [45]:
dfs_concat = {
    k: pd.concat(v, ignore_index=True)
    for k, v in dfs.items()
}

🔹 7. Print por pipeline

In [46]:
def print_pipeline_summary(name, df):
    accuracy = df["correct"].mean()
    evidence_score = df["avg_evidence_per_step"].mean()
    proxy_score = 0.7 * accuracy + 0.3 * min(evidence_score / 5.0, 1.0)

    df_errors = df[~df["correct"]]

    print("=" * 50)
    print(f"📌 PIPELINE: {name.upper()}")
    print("=" * 50)

    print('🎯 Accuracy:', round(accuracy, 3), '->', interpret(accuracy, 'accuracy'))
    print('📊 Proxy Score:', round(proxy_score, 3), '->', interpret(proxy_score, 'proxy_score'))

    print("📊 Total samples:", len(df))
    print("✅ Correct predictions:", df["correct"].sum())
    print("❌ Errors:", len(df_errors))
    print("\n")

🔹 8. Executar resumo

In [47]:
for name in dfs_concat:
    print_pipeline_summary(name, dfs_concat[name])

📌 PIPELINE: BASELINE
🎯 Accuracy: 0.0 -> ⚠️ Baixo
📊 Proxy Score: 0.0 -> ⚠️ Fraco
📊 Total samples: 159
✅ Correct predictions: 0
❌ Errors: 159


📌 PIPELINE: ITERATIVE
🎯 Accuracy: 0.0 -> ⚠️ Baixo
📊 Proxy Score: 0.0 -> ⚠️ Fraco
📊 Total samples: 221
✅ Correct predictions: 0
❌ Errors: 221


